# ReviewPulse: more seeds and more training data (Kaggle, GPU T4 x2)

Follow-up to the first fine-tuning run. That run used **10,000** training reviews and **one seed**; its three recipes were within about 1 point of each other on 1,000 validation reviews, which is inside the noise. This notebook answers two questions:

1. **Seeds:** how much does the result move when only the random seed changes? (3 seeds per setting)
2. **Data:** how much does more training data help? (**10,000, 20,000, 40,000** reviews, nested: the 10,000 are always included)

One recipe (`lr1e5_ep2`, the winner of the first study), 9 runs in total. The two T4s each train their own share of the runs **at the same time**. Validation (1,000) and test (600) are the same as before. Everything is logged to MLflow (experiment `reviewpulse-bert`).

**Before you run it, in the right-hand Settings panel:**
- Accelerator: **GPU T4 x2**
- Internet: **On**

**Expected time (an estimate, from 4.5 minutes per 10,000 reviews for 2 epochs on one T4):** about 50 to 60 minutes for the real run, plus a few minutes of setup. If the session dies part-way, run the real-run cell again: finished runs are skipped.

Run the cells in order. Do not skip the smoke test.

In [1]:
import os

import torch

os.environ["TOKENIZERS_PARALLELISM"] = "false"
n = torch.cuda.device_count()
print("GPUs visible:", n, [torch.cuda.get_device_name(i) for i in range(n)])
assert n == 2, "Select 'GPU T4 x2' in Settings > Accelerator, then restart the session."

GPUs visible: 2 ['Tesla T4', 'Tesla T4']


In [2]:
# 1. Get the code (main branch of the team repository) and MLflow 3.18.0
%cd /kaggle/working
!rm -rf ReviewPulse && git clone --depth 1 https://github.com/Kashish598/ReviewPulse.git
%cd ReviewPulse
!pip install -q mlflow==3.18.0

/kaggle/working
Cloning into 'ReviewPulse'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 42 (delta 0), reused 30 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 65.76 KiB | 6.58 MiB/s, done.
/kaggle/working/ReviewPulse


## 2. Add the two scripts

`src/finetune_bert.py` is the first study's script (the new one reuses its training code). `src/finetune_scaling.py` is new. Once both are committed to `main`, you can delete these two cells.

In [3]:
%%writefile src/finetune_bert.py
"""Fine-tune the nlptown BERT on the 10,000 training reviews and track everything in MLflow.

Why this design
---------------
* Start from the model we already selected: `nlptown/bert-base-multilingual-uncased-sentiment`
  (1 to 5 stars). Its zero-shot test accuracy on the 600 reviews is the number to beat.
* Keep the 5-star output layer. The training labels are only negative/positive, so the loss
  is computed on the *mapped* probabilities: P(negative) = P(1) + P(2) + P(3) and
  P(positive) = P(4) + P(5), the same mapping the API uses. Training therefore starts from the
  zero-shot model instead of from a randomly initialised head, and the saved model is a
  drop-in replacement for `models/nlptown_sentiment` (same labels, same `BertPredictor`).
* Predictions use the same rule as the API: take the most likely star, then map it.
  Metrics from this script are therefore the ones the API will reproduce.
* Three recipes are compared on the validation split only. The best one (macro-F1, accuracy
  breaks ties) is saved and scored on the test split exactly once. No test result is used
  to choose anything.

MLflow (experiment `reviewpulse-bert`, tagged stage=finetune_candidate / finetune_selected)
-------------------------------------------------------------------------------------------
One run per recipe (parameters, per-epoch validation metrics, best epoch, timing) and one
`finetune_selected_<recipe>` run with validation and test metrics plus the saved metadata.

Run (GPU strongly recommended, see README):
    python -m src.finetune_bert --smoke           # tiny end-to-end check, no MLflow, own outputs
    python -m src.finetune_bert                   # all three recipes on the real splits
    python -m src.finetune_bert --configs lr2e5_ep3
"""

from __future__ import annotations

import argparse
import contextlib
import gc
import json
import math
import os
import random
import shutil
import time
from dataclasses import asdict, dataclass, replace
from datetime import UTC, datetime

import numpy as np

from src.common import (
    DATASET_REVISION,
    MLFLOW_BERT_EXPERIMENT,
    MODELS_DIR,
    PREDICTIONS_DIR,
    SEED,
    compute_metrics,
    load_split,
    write_predictions,
)

BASE_MODEL = "nlptown/bert-base-multilingual-uncased-sentiment"
FINETUNED_DIR = MODELS_DIR / "nlptown_finetuned"
DEFAULT_TRACKING_URI = "sqlite:///reviewpulse_mlflow.db"

# Index 0 is "1 star", index 4 is "5 stars". 1 to 3 stars -> negative (0), 4 to 5 -> positive (1).
STAR_TO_BINARY = [0, 0, 0, 1, 1]
EVAL_MAX_LENGTH = 512  # same truncation length as the API and the zero-shot evaluation


@dataclass(frozen=True)
class FinetuneConfig:
    """One fine-tuning recipe. `freeze_layers` freezes the embeddings and that many encoder layers."""

    name: str
    learning_rate: float
    epochs: int
    batch_size: int = 16
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    freeze_layers: int = 0
    max_length: int = 256


CONFIGS = {
    config.name: config
    for config in (
        # Small steps, short run: least risk of forgetting what the model already knows.
        FinetuneConfig("lr1e5_ep2", learning_rate=1e-5, epochs=2),
        # The usual BERT recipe.
        FinetuneConfig("lr2e5_ep3", learning_rate=2e-5, epochs=3),
        # Train only the upper half of the network, with a larger step size.
        FinetuneConfig("lr3e5_ep3_freeze6", learning_rate=3e-5, epochs=3, freeze_layers=6),
    )
}


# ------------------------------------------------------------- pure helpers (no torch)


def pick_best(results: dict[str, dict]) -> str:
    """Name of the recipe with the best validation macro-F1 (accuracy breaks ties)."""
    if not results:
        raise ValueError("No results to choose from.")
    return max(results, key=lambda name: (results[name]["macro_f1"], results[name]["accuracy"]))


def decide(probabilities: np.ndarray) -> tuple[list[int], list[float]]:
    """Serving rule: most likely star, mapped to 0/1. Score = probability of that star."""
    probabilities = np.asarray(probabilities)
    stars = probabilities.argmax(axis=1)
    labels = [STAR_TO_BINARY[int(star)] for star in stars]
    scores = [float(score) for score in probabilities.max(axis=1)]
    return labels, scores


def decide_by_sum(probabilities: np.ndarray) -> list[int]:
    """Alternative rule: positive when P(4) + P(5) > 0.5. Logged as a diagnostic only."""
    probabilities = np.asarray(probabilities)
    return [int(value > 0.5) for value in probabilities[:, 3:].sum(axis=1)]


def evaluate_probabilities(probabilities: np.ndarray, true_labels: list[int]) -> dict:
    """Metrics for a (N, 5) probability matrix, using the serving rule."""
    labels, _ = decide(probabilities)
    metrics = compute_metrics(true_labels, labels)
    metrics["accuracy_sum_rule"] = compute_metrics(true_labels, decide_by_sum(probabilities))[
        "accuracy"
    ]
    return metrics


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
    except ImportError:
        return
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ------------------------------------------------------------------------ MLflow wrapper


class Tracker:
    """Thin MLflow wrapper. When disabled every call does nothing, so the script runs anywhere."""

    def __init__(self, enabled: bool, tracking_uri: str | None = None):
        self.enabled = enabled
        if enabled:
            import mlflow

            mlflow.set_tracking_uri(tracking_uri or DEFAULT_TRACKING_URI)
            mlflow.set_experiment(MLFLOW_BERT_EXPERIMENT)

    @contextlib.contextmanager
    def run(self, run_name: str, tags: dict | None = None):
        if not self.enabled:
            yield self
            return
        import mlflow

        with mlflow.start_run(run_name=run_name):
            mlflow.set_tags(tags or {})
            yield self

    def params(self, values: dict) -> None:
        if self.enabled:
            import mlflow

            mlflow.log_params({key: str(value) for key, value in values.items()})

    def metrics(self, values: dict, step: int | None = None) -> None:
        if self.enabled:
            import mlflow

            mlflow.log_metrics({k: float(v) for k, v in values.items()}, step=step)

    def dict(self, content: dict, file_name: str) -> None:
        if self.enabled:
            import mlflow

            mlflow.log_dict(content, file_name)

    def artifact(self, path) -> None:
        if self.enabled:
            import mlflow

            mlflow.log_artifact(str(path))


# ------------------------------------------------------------------- torch helpers


def binary_log_probs(logits):
    """(N, 5) star logits -> (N, 2) log-probabilities of [negative, positive]."""
    import torch

    log_p = torch.log_softmax(logits.float(), dim=-1)
    negative = torch.logsumexp(log_p[:, :3], dim=-1)
    positive = torch.logsumexp(log_p[:, 3:], dim=-1)
    return torch.stack([negative, positive], dim=-1)


def training_loss(logits, labels):
    """Negative log-likelihood of the true binary label under the mapped probabilities."""
    from torch.nn import functional

    return functional.nll_loss(binary_log_probs(logits), labels)


def freeze_lower_layers(model, n_layers: int) -> None:
    """Freeze the embeddings and the first `n_layers` encoder layers."""
    if n_layers <= 0:
        return
    for parameter in model.bert.embeddings.parameters():
        parameter.requires_grad = False
    for layer in model.bert.encoder.layer[:n_layers]:
        for parameter in layer.parameters():
            parameter.requires_grad = False


def count_parameters(model) -> tuple[int, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def _autocast(torch, enabled: bool):
    if enabled:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return contextlib.nullcontext()


def predict_probabilities(model, tokenizer, texts, device, use_amp, batch_size=64):
    """Softmax probabilities over the 5 stars, in the original order of `texts`."""
    import torch

    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))  # similar lengths per batch
    output = np.zeros((len(texts), 5), dtype=np.float32)
    model.eval()
    with torch.no_grad():
        for start in range(0, len(order), batch_size):
            index = order[start : start + batch_size]
            batch = tokenizer(
                [texts[i] for i in index],
                truncation=True,
                max_length=EVAL_MAX_LENGTH,
                padding=True,
                return_tensors="pt",
            ).to(device)
            with _autocast(torch, use_amp):
                logits = model(**batch).logits
            output[index] = torch.softmax(logits.float(), dim=-1).cpu().numpy()
    return output


def share_truncated(tokenizer, texts: list[str], max_length: int) -> float:
    """Fraction of reviews longer than `max_length` tokens (they are cut during training)."""
    lengths = tokenizer(texts, truncation=False, verbose=False)["input_ids"]
    return float(np.mean([len(ids) > max_length for ids in lengths]))


def make_scaler(torch, enabled: bool):
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        return torch.amp.GradScaler("cuda", enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)


# -------------------------------------------------------------------------- training


def train_one(config, train, validation, tokenizer, device, use_amp, seed, tracker, zero_shot):
    """Fine-tune one recipe. Returns the best-epoch model and a summary dict."""
    import torch
    from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL).to(device)
    freeze_lower_layers(model, config.freeze_layers)
    total_params, trainable_params = count_parameters(model)

    named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    no_decay = ("bias", "LayerNorm.weight")
    optimizer = torch.optim.AdamW(
        [
            {
                "params": [p for n, p in named if not any(k in n for k in no_decay)],
                "weight_decay": config.weight_decay,
            },
            {
                "params": [p for n, p in named if any(k in n for k in no_decay)],
                "weight_decay": 0.0,
            },
        ],
        lr=config.learning_rate,
    )
    steps_per_epoch = math.ceil(len(train) / config.batch_size)
    total_steps = steps_per_epoch * config.epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(total_steps * config.warmup_ratio), total_steps
    )
    scaler = make_scaler(torch, use_amp)

    texts = train["review_text"].tolist()
    labels = train["label"].tolist()
    validation_texts = validation["review_text"].tolist()
    validation_labels = validation["label"].tolist()

    tracker.params(
        {
            **asdict(config),
            "base_model": BASE_MODEL,
            "seed": seed,
            "train_size": len(train),
            "validation_size": len(validation),
            "dataset_revision": DATASET_REVISION,
            "device": str(device),
            "mixed_precision": use_amp,
            "eval_max_length": EVAL_MAX_LENGTH,
            "total_parameters": total_params,
            "trainable_parameters": trainable_params,
            "train_share_truncated": round(share_truncated(tokenizer, texts, config.max_length), 4),
            "loss": "nll on mapped binary probabilities (1-3 stars vs 4-5 stars)",
        }
    )
    tracker.metrics(
        {
            "val_accuracy": zero_shot["accuracy"],
            "val_macro_f1": zero_shot["macro_f1"],
            "val_accuracy_sum_rule": zero_shot["accuracy_sum_rule"],
        },
        step=0,
    )

    best = {"macro_f1": -1.0, "accuracy": -1.0, "epoch": 0, "state": None}
    started = time.perf_counter()
    global_step = 0
    for epoch in range(1, config.epochs + 1):
        model.train()
        order = np.random.default_rng(seed + epoch).permutation(len(texts))
        epoch_started = time.perf_counter()
        losses = []
        for start in range(0, len(order), config.batch_size):
            index = order[start : start + config.batch_size]
            batch = tokenizer(
                [texts[i] for i in index],
                truncation=True,
                max_length=config.max_length,
                padding=True,
                return_tensors="pt",
            ).to(device)
            target = torch.tensor([labels[i] for i in index], device=device)

            with _autocast(torch, use_amp):
                logits = model(**batch).logits
            loss = training_loss(logits, target)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for _, p in named], 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            global_step += 1
            losses.append(float(loss.detach()))
            if global_step % 50 == 0:
                tracker.metrics({"train_loss_step": float(np.mean(losses[-50:]))}, global_step)

        probabilities = predict_probabilities(model, tokenizer, validation_texts, device, use_amp)
        metrics = evaluate_probabilities(probabilities, validation_labels)
        tracker.metrics(
            {
                "train_loss": float(np.mean(losses)),
                "val_accuracy": metrics["accuracy"],
                "val_macro_f1": metrics["macro_f1"],
                "val_accuracy_sum_rule": metrics["accuracy_sum_rule"],
                "epoch_seconds": time.perf_counter() - epoch_started,
            },
            step=epoch,
        )
        print(
            f"  epoch {epoch}/{config.epochs}: train loss {np.mean(losses):.4f}, "
            f"validation accuracy {metrics['accuracy']:.4f}, macro-F1 {metrics['macro_f1']:.4f}"
        )

        if (metrics["macro_f1"], metrics["accuracy"]) > (best["macro_f1"], best["accuracy"]):
            best = {
                "macro_f1": metrics["macro_f1"],
                "accuracy": metrics["accuracy"],
                "epoch": epoch,
                "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "metrics": metrics,
            }

    model.load_state_dict(best["state"])
    summary = {
        "macro_f1": best["macro_f1"],
        "accuracy": best["accuracy"],
        "best_epoch": best["epoch"],
        "metrics": best["metrics"],
        "train_seconds": time.perf_counter() - started,
        "trainable_parameters": trainable_params,
    }
    tracker.metrics(
        {
            "best_val_accuracy": summary["accuracy"],
            "best_val_macro_f1": summary["macro_f1"],
            "best_epoch": summary["best_epoch"],
            "train_seconds": summary["train_seconds"],
        }
    )
    tracker.dict(summary["metrics"], "validation_metrics.json")
    return model, summary


def release() -> None:
    """Free memory after the caller has deleted a model (call `del model` first)."""
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass


# ----------------------------------------------------------------------------- main


def parse_args():
    parser = argparse.ArgumentParser(description="Fine-tune nlptown BERT and track it in MLflow.")
    parser.add_argument(
        "--configs",
        nargs="+",
        default=list(CONFIGS),
        choices=list(CONFIGS),
        help="recipes to run (default: all three)",
    )
    parser.add_argument(
        "--smoke",
        action="store_true",
        help="tiny end-to-end check: a few reviews, 1 epoch, no MLflow, separate output files",
    )
    parser.add_argument("--no-mlflow", action="store_true", help="skip MLflow logging")
    parser.add_argument(
        "--tracking-uri",
        default=os.environ.get("MLFLOW_TRACKING_URI", DEFAULT_TRACKING_URI),
        help="MLflow database (default: $MLFLOW_TRACKING_URI, else a local sqlite file)",
    )
    parser.add_argument("--seed", type=int, default=SEED)
    parser.add_argument("--no-amp", action="store_true", help="disable fp16 mixed precision")
    return parser.parse_args()


def main():
    args = parse_args()

    import torch
    import transformers
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda" and not args.no_amp
    if device.type == "cpu":
        print("WARNING: no GPU found. Fine-tuning BERT on a CPU takes many hours. Use --smoke")
        print("         to check the script, and run the real training on a GPU (Kaggle, Colab).")

    suffix = "_smoke" if args.smoke else ""
    model_dir = FINETUNED_DIR.with_name(FINETUNED_DIR.name + suffix)
    use_mlflow = not (args.no_mlflow or args.smoke)
    configs = {
        n: (replace(CONFIGS[n], epochs=1) if args.smoke else CONFIGS[n]) for n in args.configs
    }
    train = load_split("train")
    validation = load_split("validation")
    if args.smoke:
        print("Smoke mode: a few reviews, 1 epoch, no MLflow. The scores mean nothing.")
        train, validation = train.head(160), validation.head(80)
    print(
        f"Train: {len(train)} reviews. Validation: {len(validation)} reviews. "
        f"Device: {device}, mixed precision: {use_amp}."
    )

    tracker = Tracker(use_mlflow, args.tracking_uri)
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

    # Zero-shot reference on validation: it should be close to 0.930 accuracy / 0.9298 macro-F1,
    # the numbers from the Kaggle candidate comparison. A big difference means a setup problem.
    print("Scoring the pretrained model on validation (reference)...")
    reference = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL).to(device)
    zero_shot = evaluate_probabilities(
        predict_probabilities(
            reference, tokenizer, validation["review_text"].tolist(), device, use_amp
        ),
        validation["label"].tolist(),
    )
    del reference
    release()
    print(
        f"  zero-shot validation accuracy {zero_shot['accuracy']:.4f}, "
        f"macro-F1 {zero_shot['macro_f1']:.4f}"
    )

    results: dict[str, dict] = {}
    best_name = None
    base_tags = {"model_family": "bert", "base_model": BASE_MODEL, "head": "nlptown_5_star_mapped"}
    for name, config in configs.items():
        print(f"\nRecipe {name}: {asdict(config)}")
        with tracker.run(
            f"finetune_{name}",
            {**base_tags, "stage": "finetune_candidate", "transformers": transformers.__version__},
        ):
            model, summary = train_one(
                config, train, validation, tokenizer, device, use_amp, args.seed, tracker, zero_shot
            )
        results[name] = summary

        if pick_best(results) == name:  # new best so far: keep these weights on disk
            if model_dir.exists():
                shutil.rmtree(model_dir)
            model_dir.mkdir(parents=True)
            model.save_pretrained(model_dir)
            tokenizer.save_pretrained(model_dir)
            best_name = name
            print(f"  new best on validation, saved to {model_dir}")
        del model
        release()

    print(f"\nSelected on validation: {best_name}")
    chosen = configs[best_name]
    chosen_summary = results[best_name]

    metadata = {
        "model_name": BASE_MODEL,
        "label_mapping": "1-3 stars -> negative, 4-5 stars -> positive",
        "max_token_length": EVAL_MAX_LENGTH,
        "fine_tuned": True,
        "recipe": asdict(chosen),
        "training_loss": "nll on mapped binary probabilities (1-3 stars vs 4-5 stars)",
        "best_epoch": chosen_summary["best_epoch"],
        "train_size": len(train),
        "dataset_revision": DATASET_REVISION,
        "seed": args.seed,
        "created_at": datetime.now(UTC).isoformat(timespec="seconds"),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "validation_accuracy": chosen_summary["accuracy"],
        "validation_macro_f1": chosen_summary["macro_f1"],
    }
    metadata_path = model_dir / "metadata.json"
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    # Score the held-out test split once, through the saved artifact.
    test = load_split("test")
    if args.smoke:
        test = test.head(80)
    saved = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
    probabilities = predict_probabilities(
        saved, tokenizer, test["review_text"].tolist(), device, use_amp
    )
    del saved
    release()
    labels, scores = decide(probabilities)
    test_metrics = compute_metrics(test["label"].tolist(), labels)
    test_metrics["model"] = f"nlptown-finetuned-{best_name}"
    test_metrics["split"] = "test"

    PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
    predictions_path = write_predictions(
        PREDICTIONS_DIR / f"bert_finetuned{suffix}_test.csv",
        test["row_id"].tolist(),
        test["label"].tolist(),
        labels,
        scores,
    )
    metrics_path = PREDICTIONS_DIR / f"bert_finetuned{suffix}_test_metrics.json"
    metrics_path.write_text(json.dumps(test_metrics, indent=4), encoding="utf-8")
    metadata["test_accuracy"] = test_metrics["accuracy"]
    metadata["test_macro_f1"] = test_metrics["macro_f1"]
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    with tracker.run(
        f"finetune_selected_{best_name}",
        {**base_tags, "stage": "finetune_selected", "transformers": transformers.__version__},
    ):
        tracker.params(
            {
                **asdict(chosen),
                "base_model": BASE_MODEL,
                "seed": args.seed,
                "train_size": len(train),
                "validation_size": len(validation),
                "test_size": len(test),
                "dataset_revision": DATASET_REVISION,
                "artifact_path": str(model_dir),
            }
        )
        tracker.metrics(
            {
                "val_accuracy": chosen_summary["accuracy"],
                "val_macro_f1": chosen_summary["macro_f1"],
                "test_accuracy": test_metrics["accuracy"],
                "test_macro_f1": test_metrics["macro_f1"],
                "zero_shot_val_accuracy": zero_shot["accuracy"],
                "zero_shot_val_macro_f1": zero_shot["macro_f1"],
                "best_epoch": chosen_summary["best_epoch"],
            }
        )
        tracker.dict(chosen_summary["metrics"], "validation_metrics.json")
        for path in (metadata_path, metrics_path, predictions_path):
            tracker.artifact(path)

    print("\nValidation results")
    print(
        f"  {'zero-shot (pretrained)':<22} accuracy {zero_shot['accuracy']:.4f}  "
        f"macro-F1 {zero_shot['macro_f1']:.4f}"
    )
    for name, summary in results.items():
        print(
            f"  {name:<22} accuracy {summary['accuracy']:.4f}  macro-F1 {summary['macro_f1']:.4f}"
            f"  (best epoch {summary['best_epoch']}, {summary['train_seconds'] / 60:.1f} min)"
        )

    print(f"\nHeld-out test results for {test_metrics['model']}")
    print(f"  accuracy {test_metrics['accuracy']:.4f}, macro-F1 {test_metrics['macro_f1']:.4f}")
    print(f"  confusion matrix: {test_metrics['confusion_matrix']}")
    zero_shot_file = PREDICTIONS_DIR / "bert_test_metrics.json"
    if zero_shot_file.exists() and not args.smoke:
        before = json.loads(zero_shot_file.read_text(encoding="utf-8"))
        print(
            f"  pretrained model on the same test set: accuracy {before['accuracy']:.4f}, "
            f"macro-F1 {before['macro_f1']:.4f}"
        )
    print(f"Predictions saved to: {predictions_path}")
    print(f"Model stored in {model_dir}/ (kept out of Git).")
    if use_mlflow:
        print(f"MLflow experiment: {MLFLOW_BERT_EXPERIMENT} at {args.tracking_uri}")


if __name__ == "__main__":
    main()

Writing src/finetune_bert.py


In [4]:
%%writefile src/finetune_scaling.py
"""Seeds and training-set size for the fine-tuned nlptown BERT, tracked in MLflow.

Why
---
`src/finetune_bert.py` trained on 10,000 reviews with one seed. Its three recipes were within
about one point of each other on 1,000 validation reviews, which is inside the noise of that
set, and one seed cannot say whether 0.944 is typical or lucky. This script asks two questions:

  1. How much does the result move between seeds?      (3 seeds per setting)
  2. How much does more training data help?              (10,000, 20,000 and 40,000 reviews)

Design
------
* One recipe (default `lr1e5_ep2`, the winner of the first study) and the same training code
  (`finetune_bert.train_one`): same loss, same 5-star head, same serving rule.
* Validation (1,000) and test (600) are exactly the old splits. The extra training reviews come
  from the rest of the official train pool, with duplicates removed and anything that appears in
  validation or test excluded. The 10,000 original reviews are always the first rows, so the
  training sets are nested: 10,000 inside 20,000 inside 40,000.
* Every (size, seed) pair is one run. Runs are split between workers (one per GPU), so two
  T4s train two runs at the same time.
* Workers never talk to MLflow. They record what they would have logged in a JSON file next to
  the results, and the `summarize` step replays everything into MLflow from one process. That
  avoids two processes writing to one sqlite file, and a logging problem cannot kill a
  20-minute training run.
* Selection uses validation only (macro-F1, accuracy breaks ties). The test set is scored for
  every run so the spread between seeds can be reported, but it is never used to choose
  anything. Quote the MEAN test accuracy over seeds, not the best seed: the best seed was
  picked on validation and is a slightly optimistic pick.

Commands (from the repository root)
-----------------------------------
    python -m src.finetune_scaling prepare                       # write train_extended.csv
    python -m src.finetune_scaling train --worker 0 --workers 2  # GPU 0 (set CUDA_VISIBLE_DEVICES)
    python -m src.finetune_scaling train --worker 1 --workers 2  # GPU 1, at the same time
    python -m src.finetune_scaling summarize                     # select, re-score, log to MLflow
    python -m src.finetune_scaling train --smoke ; ... summarize --smoke   # tiny end-to-end check
"""

from __future__ import annotations

import argparse
import contextlib
import json
import math
import os
import shutil
import time
from dataclasses import asdict, dataclass, replace
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import pandas as pd

from src.common import (
    DATA_COLUMNS,
    DATASET_REVISION,
    MLFLOW_BERT_EXPERIMENT,
    MODELS_DIR,
    PREDICTIONS_DIR,
    PROCESSED_DIR,
    SEED,
    compute_metrics,
    load_split,
    read_predictions,
    write_predictions,
)
from src.finetune_bert import (
    BASE_MODEL,
    CONFIGS,
    DEFAULT_TRACKING_URI,
    EVAL_MAX_LENGTH,
    Tracker,
    decide,
    evaluate_probabilities,
    pick_best,
    predict_probabilities,
    release,
    train_one,
)

DEFAULT_RECIPE = "lr1e5_ep2"
DEFAULT_SIZES = (10_000, 20_000, 40_000)
DEFAULT_SEEDS = (42, 43, 44)
SMOKE_SIZES = (80, 160)
SMOKE_SEEDS = (42, 43)
SMOKE_EVAL_ROWS = 80
EXTENDED_TRAIN_FILE = "train_extended.csv"


@dataclass(frozen=True)
class Layout:
    """Where one study keeps its files. The smoke test gets its own folders."""

    results_dir: Path
    runs_dir: Path
    final_model_dir: Path
    suffix: str


def layout(smoke: bool) -> Layout:
    suffix = "_smoke" if smoke else ""
    return Layout(
        results_dir=PREDICTIONS_DIR / f"scaling{suffix}",
        runs_dir=MODELS_DIR / f"scaling_runs{suffix}",
        final_model_dir=MODELS_DIR / f"nlptown_finetuned_v2{suffix}",
        suffix=suffix,
    )


# ------------------------------------------------------------- pure helpers (no torch)


def run_name(size: int, seed: int) -> str:
    return f"scale_n{size}_seed{seed}"


def make_jobs(sizes, seeds) -> list[tuple[int, int]]:
    """Every (training size, seed) pair."""
    sizes, seeds = [int(s) for s in sizes], [int(s) for s in seeds]
    if not sizes or not seeds:
        raise ValueError("Need at least one size and one seed.")
    if len(set(sizes)) != len(sizes) or len(set(seeds)) != len(seeds):
        raise ValueError("Sizes and seeds must not repeat.")
    if min(sizes) < 1:
        raise ValueError("Sizes must be positive.")
    return [(size, seed) for size in sizes for seed in seeds]


def assign_jobs(jobs, n_workers: int) -> list[list[tuple[int, int]]]:
    """Split jobs between workers so the total number of training reviews is balanced.

    Greedy: biggest job first, always to the least loaded worker. Inside a worker the jobs run
    from small to large, so the first results (a sanity check) arrive early.
    """
    if n_workers < 1:
        raise ValueError("Need at least one worker.")
    loads = [0] * n_workers
    assigned: list[list[tuple[int, int]]] = [[] for _ in range(n_workers)]
    for job in sorted(jobs, key=lambda j: (-j[0], j[1])):
        worker = loads.index(min(loads))
        assigned[worker].append(job)
        loads[worker] += job[0]
    return [sorted(worker_jobs) for worker_jobs in assigned]


def extend_train(
    base: pd.DataFrame,
    pool: pd.DataFrame,
    validation: pd.DataFrame,
    test: pd.DataFrame,
    n_total: int,
    seed: int = SEED,
) -> pd.DataFrame:
    """The original training reviews first, then extra reviews from the rest of the train pool.

    `pool` is the cleaned official train split (see `prepare_data.clean_reviews`). Extra rows
    never include anything from `base`, `validation` or `test`, and are a random sample, so
    taking the first k rows of the result always gives a nested training set.
    """
    if n_total < len(base):
        raise ValueError(f"n_total ({n_total}) is smaller than the base split ({len(base)}).")

    pool_text = pool.set_index("row_id")["review_text"]
    base_text = base.set_index("row_id")["review_text"]
    if not base_text.index.isin(pool_text.index).all() or not (
        pool_text.reindex(base_text.index) == base_text
    ).all():
        raise ValueError(
            "The base training split does not come from this pool. "
            "Check that the dataset revision and the seed are the ones used for the splits."
        )

    banned = set(base["review_text"]) | set(validation["review_text"]) | set(test["review_text"])
    rest = pool[~pool["review_text"].isin(banned) & ~pool["row_id"].isin(base["row_id"])]
    need = n_total - len(base)
    if len(rest) < need:
        raise ValueError(f"Only {len(rest)} extra reviews are available, need {need}.")

    extra = rest.sample(n=need, random_state=seed)
    result = pd.concat([base[DATA_COLUMNS], extra[DATA_COLUMNS]], ignore_index=True)
    if not result["review_text"].is_unique or not result["row_id"].is_unique:
        raise ValueError("The extended training set contains duplicates.")
    return result


def exact_mcnemar_p(fixed: int, broken: int) -> float:
    """Two-sided exact McNemar test from the two discordant counts."""
    n = fixed + broken
    if n == 0:
        return 1.0
    k = min(fixed, broken)
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2**n)


def wilson_interval(correct: int, n: int, z: float = 1.959964) -> tuple[float, float]:
    """95% Wilson score interval for an accuracy."""
    if n <= 0:
        raise ValueError("n must be positive.")
    p = correct / n
    denominator = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denominator
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denominator
    return centre - half, centre + half


def compare_predictions(path_a, path_b) -> dict | None:
    """Paired comparison of two prediction files on the same reviews (A is the reference).

    `fixed` = reviews A gets wrong and B gets right, `broken` = the other way round.
    Returns None when either file is missing.
    """
    if not Path(path_a).exists() or not Path(path_b).exists():
        return None
    a = read_predictions(path_a).set_index("row_id")
    b = read_predictions(path_b).set_index("row_id")
    if set(a.index) != set(b.index):
        raise ValueError("The two prediction files do not cover the same reviews.")
    b = b.loc[a.index]
    if not (a["label_true"] == b["label_true"]).all():
        raise ValueError("The two prediction files disagree on the true labels.")
    a_right = (a["pred"] == a["label_true"]).to_numpy()
    b_right = (b["pred"] == b["label_true"]).to_numpy()
    fixed = int((~a_right & b_right).sum())
    broken = int((a_right & ~b_right).sum())
    return {
        "n": int(len(a)),
        "accuracy_a": float(a_right.mean()),
        "accuracy_b": float(b_right.mean()),
        "fixed": fixed,
        "broken": broken,
        "mcnemar_p": exact_mcnemar_p(fixed, broken),
    }


def _json_default(value):
    """numpy numbers become plain numbers; anything else becomes text."""
    return value.item() if hasattr(value, "item") else str(value)


class RecordingTracker:
    """Same interface as `Tracker`, but only remembers the calls (plain dicts, JSON-safe)."""

    enabled = True

    def __init__(self):
        self.calls: list[dict] = []

    @contextlib.contextmanager
    def run(self, run_name, tags=None):
        yield self

    def params(self, values: dict) -> None:
        self.calls.append({"op": "params", "values": {k: str(v) for k, v in values.items()}})

    def metrics(self, values: dict, step: int | None = None) -> None:
        self.calls.append(
            {"op": "metrics", "values": {k: float(v) for k, v in values.items()}, "step": step}
        )

    def dict(self, content: dict, file_name: str) -> None:
        self.calls.append(
            {
                "op": "dict",
                "content": json.loads(json.dumps(content, default=_json_default)),
                "file": file_name,
            }
        )

    def artifact(self, path) -> None:
        pass  # files are not copied; the summarize step logs the files it produces


def replay(calls: list[dict], tracker) -> None:
    """Send recorded calls to a real `Tracker` (inside an open run)."""
    for call in calls:
        if call["op"] == "params":
            tracker.params(call["values"])
        elif call["op"] == "metrics":
            tracker.metrics(call["values"], step=call["step"])
        elif call["op"] == "dict":
            tracker.dict(call["content"], call["file"])
        else:
            raise ValueError(f"Unknown recorded call {call['op']!r}.")


def load_summaries(results_dir: Path) -> list[dict]:
    """All finished runs, sorted by size then seed."""
    summaries = [
        json.loads(path.read_text(encoding="utf-8"))
        for path in sorted(Path(results_dir).glob("*_summary.json"))
    ]
    return sorted(summaries, key=lambda s: (s["train_size"], s["seed"]))


def select_run(summaries: list[dict]) -> str:
    """Name of the run with the best validation macro-F1 (accuracy breaks ties)."""
    return pick_best({s["name"]: s["val"] for s in summaries})


def _mean_std(values: list[float]) -> tuple[float, float | None]:
    mean = float(np.mean(values))
    return mean, (float(np.std(values, ddof=1)) if len(values) > 1 else None)


def aggregate_by_size(summaries: list[dict]) -> list[dict]:
    """Mean and standard deviation over seeds, one row per training size."""
    rows = []
    for size in sorted({s["train_size"] for s in summaries}):
        group = [s for s in summaries if s["train_size"] == size]
        row: dict = {"train_size": size, "runs": len(group), "seeds": [s["seed"] for s in group]}
        for split in ("val", "test"):
            for metric in ("accuracy", "macro_f1"):
                mean, std = _mean_std([s[split][metric] for s in group])
                row[f"{split}_{metric}_mean"] = mean
                row[f"{split}_{metric}_std"] = std
        accuracies = [s["test"]["accuracy"] for s in group]
        row["test_accuracy_min"], row["test_accuracy_max"] = min(accuracies), max(accuracies)
        rows.append(row)
    return rows


def ensemble_by_size(
    summaries: list[dict], results_dir: Path, val_labels: list[int], test_labels: list[int]
) -> list[dict]:
    """Average the star probabilities of all seeds of one size, then apply the serving rule.

    Diagnostic only: serving the ensemble would mean loading several models.
    """
    rows = []
    for size in sorted({s["train_size"] for s in summaries}):
        group = [s for s in summaries if s["train_size"] == size]
        if len(group) < 2:
            continue
        stored = [np.load(Path(results_dir) / f"{s['name']}_probs.npz") for s in group]
        val = evaluate_probabilities(np.mean([p["val"] for p in stored], axis=0), val_labels)
        test = evaluate_probabilities(np.mean([p["test"] for p in stored], axis=0), test_labels)
        rows.append(
            {
                "train_size": size,
                "runs": len(group),
                "val_accuracy": val["accuracy"],
                "val_macro_f1": val["macro_f1"],
                "test_accuracy": test["accuracy"],
                "test_macro_f1": test["macro_f1"],
            }
        )
    return rows


def build_report(
    summaries: list[dict], results_dir: Path, val_labels: list[int], test_labels: list[int]
) -> dict:
    """Everything the summarize step reports, computed without torch."""
    if not summaries:
        raise ValueError("No finished runs found. Run the train step first.")
    selected = select_run(summaries)
    chosen = next(s for s in summaries if s["name"] == selected)
    n_test = chosen["test"]["n"]
    low, high = wilson_interval(round(chosen["test"]["accuracy"] * n_test), n_test)
    return {
        "selected_run": selected,
        "selected_val": chosen["val"],
        "selected_test": chosen["test"],
        "selected_test_wilson_95": [low, high],
        "by_size": aggregate_by_size(summaries),
        "seed_ensembles": ensemble_by_size(summaries, results_dir, val_labels, test_labels),
    }


def runs_table(summaries: list[dict]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "run": s["name"],
                "train_size": s["train_size"],
                "seed": s["seed"],
                "best_epoch": s["best_epoch"],
                "val_accuracy": s["val"]["accuracy"],
                "val_macro_f1": s["val"]["macro_f1"],
                "test_accuracy": s["test"]["accuracy"],
                "test_macro_f1": s["test"]["macro_f1"],
                "train_minutes": round(s["train_seconds"] / 60, 1),
            }
            for s in summaries
        ]
    )


# ------------------------------------------------------------------------------ prepare


def cmd_prepare(args) -> None:
    """Build data/processed/train_extended.csv: the 10,000 original reviews plus extras."""
    from src.prepare_data import clean_reviews, load_raw

    base = load_split("train")
    validation = load_split("validation")
    test = load_split("test")
    train_raw, _, revision = load_raw(DATASET_REVISION)
    pool, cleaning = clean_reviews(train_raw, "tr")
    del train_raw
    extended = extend_train(base, pool, validation, test, args.max_size, seed=args.seed)

    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    path = PROCESSED_DIR / EXTENDED_TRAIN_FILE
    extended.to_csv(path, index=False)
    counts = {str(k): int(v) for k, v in extended["label"].value_counts().sort_index().items()}
    report = {
        "dataset_revision": revision,
        "seed": args.seed,
        "rows": len(extended),
        "base_rows": len(base),
        "extra_rows": len(extended) - len(base),
        "label_counts": counts,
        "pool_cleaning": cleaning,
    }
    (PROCESSED_DIR / "train_extended_report.json").write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2))
    print(f"Wrote {path}")


def load_training_pool(max_size: int) -> pd.DataFrame:
    """The extended file when it exists (it starts with the original rows), else the base split."""
    path = PROCESSED_DIR / EXTENDED_TRAIN_FILE
    frame = pd.read_csv(path) if path.exists() else load_split("train")
    frame["label"] = frame["label"].astype(int)
    if len(frame) < max_size:
        raise ValueError(
            f"Need {max_size} training reviews but only {len(frame)} are available. "
            "Run `python -m src.finetune_scaling prepare` first."
        )
    return frame[DATA_COLUMNS]


def load_eval_splits(smoke: bool) -> tuple[pd.DataFrame, pd.DataFrame]:
    validation, test = load_split("validation"), load_split("test")
    if smoke:
        validation, test = validation.head(SMOKE_EVAL_ROWS), test.head(SMOKE_EVAL_ROWS)
    return validation, test


# -------------------------------------------------------------------------------- train


def cmd_train(args) -> None:
    import torch
    import transformers
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    if not 0 <= args.worker < args.workers:
        raise SystemExit(f"--worker must be between 0 and {args.workers - 1}.")
    lay = layout(args.smoke)
    sizes = args.sizes or (SMOKE_SIZES if args.smoke else DEFAULT_SIZES)
    seeds = args.seeds or (SMOKE_SEEDS if args.smoke else DEFAULT_SEEDS)
    config = CONFIGS[args.recipe]
    if args.smoke:
        config = replace(config, epochs=1)
    mine = assign_jobs(make_jobs(sizes, seeds), args.workers)[args.worker]
    if not mine:
        print(f"Worker {args.worker} has no jobs.")
        return

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda" and not args.no_amp
    if device.type == "cpu":
        print("WARNING: no GPU found. Use --smoke on a CPU and run the real study on a GPU.")
    pool = load_training_pool(max(size for size, _ in mine))
    validation, test = load_eval_splits(args.smoke)
    val_texts, val_labels = validation["review_text"].tolist(), validation["label"].tolist()
    test_texts, test_labels = test["review_text"].tolist(), test["label"].tolist()
    print(
        f"Worker {args.worker}/{args.workers}: recipe {config.name}, jobs {mine}. "
        f"Validation {len(validation)}, test {len(test)}. "
        f"Device {device}, mixed precision {use_amp}."
    )

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    reference = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL).to(device)
    zero_shot = evaluate_probabilities(
        predict_probabilities(reference, tokenizer, val_texts, device, use_amp), val_labels
    )
    del reference
    release()
    print(
        f"  zero-shot validation accuracy {zero_shot['accuracy']:.4f}, "
        f"macro-F1 {zero_shot['macro_f1']:.4f}"
    )

    lay.results_dir.mkdir(parents=True, exist_ok=True)
    for size, seed in mine:
        name = run_name(size, seed)
        summary_path = lay.results_dir / f"{name}_summary.json"
        if summary_path.exists() and not args.force:
            print(f"\n{name}: already done, skipping (use --force to redo).")
            continue

        print(f"\n{name}: {size} training reviews, seed {seed}")
        train = pool.head(size).reset_index(drop=True)
        recorder = RecordingTracker()
        model, summary = train_one(
            config, train, validation, tokenizer, device, use_amp, seed, recorder, zero_shot
        )
        val_probs = predict_probabilities(model, tokenizer, val_texts, device, use_amp)
        test_probs = predict_probabilities(model, tokenizer, test_texts, device, use_amp)
        val_metrics = evaluate_probabilities(val_probs, val_labels)
        test_metrics = evaluate_probabilities(test_probs, test_labels)
        recorder.metrics(
            {"test_accuracy": test_metrics["accuracy"], "test_macro_f1": test_metrics["macro_f1"]}
        )

        run_dir = lay.runs_dir / name
        if run_dir.exists():
            shutil.rmtree(run_dir)
        run_dir.mkdir(parents=True)
        model.save_pretrained(run_dir)
        tokenizer.save_pretrained(run_dir)
        del model
        release()

        np.savez_compressed(
            lay.results_dir / f"{name}_probs.npz",
            val=val_probs.astype(np.float32),
            test=test_probs.astype(np.float32),
        )
        labels, scores = decide(test_probs)
        write_predictions(
            lay.results_dir / f"{name}_test.csv",
            test["row_id"].tolist(),
            test_labels,
            labels,
            scores,
        )
        record = {
            "name": name,
            "train_size": len(train),
            "seed": seed,
            "recipe": asdict(config),
            "best_epoch": summary["best_epoch"],
            "train_seconds": summary["train_seconds"],
            "val": val_metrics,
            "test": test_metrics,
            "model_dir": str(run_dir),
            "created_at": datetime.now(UTC).isoformat(timespec="seconds"),
            "torch": torch.__version__,
            "transformers": transformers.__version__,
            "log": recorder.calls,
        }
        # Written last: a run without its summary file counts as not done and is redone.
        text = json.dumps(record, indent=2, default=_json_default)
        summary_path.write_text(text, encoding="utf-8")
        print(
            f"  {name}: validation accuracy {val_metrics['accuracy']:.4f} "
            f"(training-time best {summary['accuracy']:.4f}), "
            f"test accuracy {test_metrics['accuracy']:.4f}, "
            f"{summary['train_seconds'] / 60:.1f} min"
        )
    print(f"\nWorker {args.worker} finished.")


# --------------------------------------------------------------------------- summarize


def _print_report(report: dict, summaries: list[dict], comparisons: dict) -> None:
    print("\nAll runs")
    print(runs_table(summaries).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

    print("\nBy training size (mean +/- std over seeds)")
    for row in report["by_size"]:

        def fmt(key, row=row):
            std = row[f"{key}_std"]
            return f"{row[f'{key}_mean']:.4f}" + (f" +/- {std:.4f}" if std is not None else "")

        print(
            f"  {row['train_size']:>6} reviews, {row['runs']} seeds: "
            f"validation accuracy {fmt('val_accuracy')}, test accuracy {fmt('test_accuracy')} "
            f"(range {row['test_accuracy_min']:.4f} to {row['test_accuracy_max']:.4f})"
        )
    if report["seed_ensembles"]:
        print("\nSeed ensembles (average of the seeds' probabilities, diagnostic only)")
        for row in report["seed_ensembles"]:
            print(
                f"  {row['train_size']:>6} reviews: validation accuracy {row['val_accuracy']:.4f}, "
                f"test accuracy {row['test_accuracy']:.4f}"
            )

    low, high = report["selected_test_wilson_95"]
    print(f"\nSelected on validation: {report['selected_run']}")
    print(
        f"  validation accuracy {report['selected_val']['accuracy']:.4f}, "
        f"test accuracy {report['selected_test']['accuracy']:.4f} "
        f"(95% interval {low:.4f} to {high:.4f}), "
        f"confusion matrix {report['selected_test']['confusion_matrix']}"
    )
    for label, result in comparisons.items():
        if result is None:
            continue
        print(
            f"  vs {label}: {result['accuracy_a']:.4f} -> {result['accuracy_b']:.4f}, "
            f"fixed {result['fixed']}, broken {result['broken']}, "
            f"exact McNemar p = {result['mcnemar_p']:.4f}"
        )
    print("\nQuote the mean over seeds as the headline number. The selected run is the best")
    print("seed on validation, so its test score is a slightly optimistic pick.")


def cmd_summarize(args) -> None:
    import torch
    import transformers
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    lay = layout(args.smoke)
    summaries = load_summaries(lay.results_dir)
    validation, test = load_eval_splits(args.smoke)
    val_labels, test_labels = validation["label"].tolist(), test["label"].tolist()
    for s in summaries:
        if s["val"]["n"] != len(validation) or s["test"]["n"] != len(test):
            raise SystemExit(f"{s['name']} was scored on different splits than the current ones.")
    report = build_report(summaries, lay.results_dir, val_labels, test_labels)
    selected = next(s for s in summaries if s["name"] == report["selected_run"])

    # Keep the winner as the new fine-tuned model; delete the other checkpoints.
    chosen_dir = Path(selected["model_dir"])
    if not chosen_dir.exists():
        raise SystemExit(f"{chosen_dir} is missing. Re-run the train step with --force for it.")
    if lay.final_model_dir.exists():
        shutil.rmtree(lay.final_model_dir)
    lay.final_model_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(chosen_dir), str(lay.final_model_dir))
    if not args.keep_runs:
        for s in summaries:
            other = Path(s["model_dir"])
            if other != chosen_dir and other.exists():
                shutil.rmtree(other)

    # Score the test split once more through the saved artifact: this is what will be served.
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda" and not args.no_amp
    tokenizer = AutoTokenizer.from_pretrained(lay.final_model_dir)
    saved = AutoModelForSequenceClassification.from_pretrained(lay.final_model_dir).to(device)
    probabilities = predict_probabilities(
        saved, tokenizer, test["review_text"].tolist(), device, use_amp
    )
    del saved
    release()
    labels, scores = decide(probabilities)
    in_run_labels, _ = decide(np.load(lay.results_dir / f"{selected['name']}_probs.npz")["test"])
    agreement = float(np.mean(np.array(labels) == np.array(in_run_labels)))
    test_metrics = compute_metrics(test_labels, labels)
    test_metrics["model"] = f"nlptown-finetuned-v2-{selected['name']}"
    test_metrics["split"] = "test"

    PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
    predictions_path = write_predictions(
        PREDICTIONS_DIR / f"bert_finetuned_v2{lay.suffix}_test.csv",
        test["row_id"].tolist(),
        test_labels,
        labels,
        scores,
    )
    metrics_path = PREDICTIONS_DIR / f"bert_finetuned_v2{lay.suffix}_test_metrics.json"
    metrics_path.write_text(json.dumps(test_metrics, indent=4), encoding="utf-8")

    comparisons: dict = {}
    if not args.smoke:
        comparisons = {
            "pretrained model": compare_predictions(
                PREDICTIONS_DIR / "bert_test.csv", predictions_path
            ),
            "first fine-tuned model (10,000 reviews, 1 seed)": compare_predictions(
                PREDICTIONS_DIR / "bert_finetuned_test.csv", predictions_path
            ),
        }

    metadata = {
        "model_name": BASE_MODEL,
        "label_mapping": "1-3 stars -> negative, 4-5 stars -> positive",
        "max_token_length": EVAL_MAX_LENGTH,
        "fine_tuned": True,
        "recipe": selected["recipe"],
        "training_loss": "nll on mapped binary probabilities (1-3 stars vs 4-5 stars)",
        "selected_run": selected["name"],
        "train_size": selected["train_size"],
        "seed": selected["seed"],
        "best_epoch": selected["best_epoch"],
        "dataset_revision": DATASET_REVISION,
        "selection": "best validation macro-F1 over all (size, seed) runs",
        "validation_accuracy": selected["val"]["accuracy"],
        "validation_macro_f1": selected["val"]["macro_f1"],
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
        "artifact_agreement_with_training_run": agreement,
        "created_at": datetime.now(UTC).isoformat(timespec="seconds"),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
    }
    (lay.final_model_dir / "metadata.json").write_text(json.dumps(metadata, indent=2))

    table_path = lay.results_dir / "runs_table.csv"
    runs_table(summaries).to_csv(table_path, index=False)
    by_size_path = lay.results_dir / "by_size.csv"
    pd.DataFrame(report["by_size"]).to_csv(by_size_path, index=False)
    report["comparisons"] = comparisons
    report_path = lay.results_dir / "report.json"
    report_path.write_text(json.dumps(report, indent=2, default=_json_default), encoding="utf-8")

    _print_report(report, summaries, comparisons)
    print(f"\nSaved artifact agrees with the training run on {agreement:.1%} of test predictions.")
    print(f"Model: {lay.final_model_dir}/   Predictions: {predictions_path}")

    if args.smoke or args.no_mlflow:
        return
    tracker = Tracker(True, args.tracking_uri)
    base_tags = {"model_family": "bert", "base_model": BASE_MODEL, "head": "nlptown_5_star_mapped"}
    for s in summaries:
        with tracker.run(
            s["name"],
            {**base_tags, "stage": "scaling_candidate", "transformers": s["transformers"]},
        ):
            replay(s["log"], tracker)
    flat: dict[str, float] = {
        "val_accuracy": selected["val"]["accuracy"],
        "val_macro_f1": selected["val"]["macro_f1"],
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
        "artifact_agreement": agreement,
    }
    for row in report["by_size"]:
        for key, value in row.items():
            if key.startswith(("val_", "test_")) and value is not None:
                flat[f"n{row['train_size']}_{key}"] = value
    for row in report["seed_ensembles"]:
        flat[f"n{row['train_size']}_ensemble_test_accuracy"] = row["test_accuracy"]
        flat[f"n{row['train_size']}_ensemble_val_accuracy"] = row["val_accuracy"]
    for key, result in (("zero_shot", comparisons.get("pretrained model")),):
        if result:
            flat[f"{key}_fixed"], flat[f"{key}_broken"] = result["fixed"], result["broken"]
            flat[f"{key}_mcnemar_p"] = result["mcnemar_p"]
    with tracker.run(
        f"scaling_selected_{selected['name']}",
        {**base_tags, "stage": "scaling_selected", "transformers": transformers.__version__},
    ):
        tracker.params(
            {
                **selected["recipe"],
                "base_model": BASE_MODEL,
                "selected_run": selected["name"],
                "seed": selected["seed"],
                "train_size": selected["train_size"],
                "sizes": [row["train_size"] for row in report["by_size"]],
                "seeds_per_size": [row["runs"] for row in report["by_size"]],
                "dataset_revision": DATASET_REVISION,
                "artifact_path": str(lay.final_model_dir),
            }
        )
        tracker.metrics(flat)
        for path in (
            lay.final_model_dir / "metadata.json",
            metrics_path,
            predictions_path,
            table_path,
            by_size_path,
            report_path,
        ):
            tracker.artifact(path)
    print(f"MLflow experiment: {MLFLOW_BERT_EXPERIMENT} at {args.tracking_uri}")


# ------------------------------------------------------------------------------- CLI


def parse_args(argv=None):
    parser = argparse.ArgumentParser(description="Seeds and training size for fine-tuned BERT.")
    commands = parser.add_subparsers(dest="command", required=True)

    prepare = commands.add_parser("prepare", help="build data/processed/train_extended.csv")
    prepare.add_argument("--max-size", type=int, default=max(DEFAULT_SIZES))
    prepare.add_argument("--seed", type=int, default=SEED)

    train = commands.add_parser("train", help="train this worker's share of the runs")
    train.add_argument("--worker", type=int, default=0, help="index of this worker (0-based)")
    train.add_argument("--workers", type=int, default=1, help="number of workers (one per GPU)")
    train.add_argument("--recipe", default=DEFAULT_RECIPE, choices=list(CONFIGS))
    train.add_argument("--sizes", type=int, nargs="+", help=f"default {list(DEFAULT_SIZES)}")
    train.add_argument("--seeds", type=int, nargs="+", help=f"default {list(DEFAULT_SEEDS)}")
    train.add_argument("--smoke", action="store_true", help="tiny end-to-end check")
    train.add_argument("--force", action="store_true", help="redo runs that already finished")
    train.add_argument("--no-amp", action="store_true", help="disable fp16 mixed precision")

    summarize = commands.add_parser("summarize", help="select, re-score and log to MLflow")
    summarize.add_argument("--smoke", action="store_true")
    summarize.add_argument("--no-mlflow", action="store_true")
    summarize.add_argument("--keep-runs", action="store_true", help="keep every run's checkpoint")
    summarize.add_argument("--no-amp", action="store_true")
    summarize.add_argument(
        "--tracking-uri",
        default=os.environ.get("MLFLOW_TRACKING_URI", DEFAULT_TRACKING_URI),
        help="MLflow database (default: $MLFLOW_TRACKING_URI, else a local sqlite file)",
    )
    return parser.parse_args(argv)


def main(argv=None) -> None:
    args = parse_args(argv)
    started = time.perf_counter()
    {"prepare": cmd_prepare, "train": cmd_train, "summarize": cmd_summarize}[args.command](args)
    print(f"({args.command} took {(time.perf_counter() - started) / 60:.1f} min)")


if __name__ == "__main__":
    main()

Writing src/finetune_scaling.py


## 3. Build the data splits

Same dataset revision and seed as your laptop. Expected sizes: train 10,000 / validation 1,000 / test 600. Compare the MD5 hashes with `Get-FileHash -Algorithm MD5 data\processed\*.csv` on your laptop.

In [5]:
!python -m src.prepare_data
import hashlib

import pandas as pd

for name in ["train", "validation", "test"]:
    path = f"data/processed/{name}.csv"
    print(f"{name:<11} {len(pd.read_csv(path)):>6} rows   md5 {hashlib.md5(open(path, 'rb').read()).hexdigest()}")

README.md: 100%|███████████████████████████| 6.81k/6.81k [00:00<00:00, 14.6MB/s]

amazon_polarity/train-00000-of-00004.par(…): downloading bytes:  |  0.00B       
amazon_polarity/train-00000-of-00004.par(…): downloading bytes: ▏| 59.9MB, 4.66M
amazon_polarity/train-00000-of-00004.par(…): downloading bytes: ▎| 72.2MB, 6.77M
amazon_polarity/train-00000-of-00004.par(…): reconstructing file:   4%| | 11.4MB
amazon_polarity/train-00000-of-00004.par(…): reconstructing file:   5%| | 13.1MB
amazon_polarity/train-00000-of-00004.par(…): reconstructing file:   6%| | 15.4MB
amazon_polarity/train-00000-of-00004.par(…): reconstructing file:   7%| | 17.7MB
amazon_polarity/train-00000-of-00004.par(…): reconstructing file:   8%| | 20.0MB
amazon_polarity/train-00000-of-00004.par(…): reconstructing file:   9%| | 24.0MB
amazon_polarity/train-00000-of-00004.par(…): downloading bytes: ▎| 82.4MB, 7.07M
amazon_polarity/train-00000-of-00004.par(…): reconstructing file:  13%|▏| 34.4MB
amazon_polarity/train-00000

## 4. Add 30,000 extra training reviews

Takes the 10,000 original reviews and adds 30,000 more from the rest of the official train pool (duplicates removed, nothing that is in validation or test). Writes `data/processed/train_extended.csv` (40,000 rows). The first 10,000 rows are the original training set, so 10,000 is inside 20,000 is inside 40,000. This downloads and cleans the full 3.6 million review train split, so it takes a few minutes.

In [6]:
!python -m src.finetune_scaling prepare

{
  "dataset_revision": "9d9c45c18f8c3cf1b23a3c27917b60cbf28f3289",
  "seed": 42,
  "rows": 40000,
  "base_rows": 10000,
  "extra_rows": 30000,
  "label_counts": {
    "0": 19957,
    "1": 20043
  },
  "pool_cleaning": {
    "original_rows": 3600000,
    "missing_removed": 0,
    "empty_removed": 0,
    "duplicates_removed": 5217,
    "rows_after_cleaning": 3594783
  }
}
Wrote /kaggle/working/ReviewPulse/data/processed/train_extended.csv
(prepare took 0.4 min)


## 5. Smoke test (a few minutes)

Tiny end-to-end run on one GPU: 80 and 160 reviews, 2 seeds, 1 epoch, no MLflow, separate output files. The scores mean nothing. It proves that training, checkpoint saving, selection and re-scoring work. If this cell fails, stop and fix it before the real run.

In [7]:
!python -m src.finetune_scaling train --smoke
!python -m src.finetune_scaling summarize --smoke
# remove the smoke outputs so they are not mixed with the real ones
!rm -rf models/scaling_runs_smoke models/nlptown_finetuned_v2_smoke data/predictions/scaling_smoke data/predictions/bert_finetuned_v2_smoke_test*

Worker 0/1: recipe lr1e5_ep2, jobs [(80, 42), (80, 43), (160, 42), (160, 43)]. Validation 80, test 80. Device cuda, mixed precision True.
config.json: 100%|█████████████████████████████| 953/953 [00:00<00:00, 3.20MB/s]
tokenizer_config.json: 100%|██████████████████| 39.0/39.0 [00:00<00:00, 116kB/s]
vocab.txt: 100%|█████████████████████████████| 872k/872k [00:00<00:00, 79.8MB/s]
special_tokens_map.json: 100%|██████████████████| 112/112 [00:00<00:00, 336kB/s]

model.safetensors: downloading bytes:                       |  0.00B            
model.safetensors: downloading bytes: ████████▉             |  274MB, 23.6MB/s  
model.safetensors: downloading bytes: ███████████▉          |  364MB, 31.5MB/s  
model.safetensors: downloading bytes: █████████████         |  397MB, 33.7MB/s  
model.safetensors: downloading bytes: ██████████████▏       |  432MB, 37.2MB/s  
model.safetensors: reconstructing file:  69%|████  |  459MB /  669MB, 39.8MB/s  
model.safetensors: downloading bytes: █████████████

## 6. Real run: two workers, one per GPU

Worker 0 uses GPU 0 and worker 1 uses GPU 1. The 9 runs are split so both finish at about the same time. This cell prints the last lines of each worker's log every 2 minutes. Check the first finished `scale_n10000_seed42` run: its validation accuracy should be close to the **0.944** of the first study (small differences are normal: GPU arithmetic is not bit-for-bit repeatable).

If you run this cell again after a crash, finished runs are skipped. To start completely fresh, delete `data/predictions/scaling` and `models/scaling_runs` first.

In [8]:
import subprocess
import sys
import time

WORKERS = 2
procs = []
for w in range(WORKERS):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(w), "PYTHONUNBUFFERED": "1"}
    log = open(f"/kaggle/working/worker{w}.log", "a")
    command = [sys.executable, "-u", "-m", "src.finetune_scaling", "train",
               "--worker", str(w), "--workers", str(WORKERS)]
    procs.append((subprocess.Popen(command, env=env, stdout=log, stderr=subprocess.STDOUT), log))

def tail(path, n=3):
    lines = [l.rstrip() for l in open(path, errors="replace") if l.strip() and "it/s" not in l and "B/s" not in l]
    return lines[-n:]

while any(p.poll() is None for p, _ in procs):
    time.sleep(120)
    for w in range(WORKERS):
        print(f"[worker {w}]", *tail(f"/kaggle/working/worker{w}.log"), sep="\n   ")
    print("-" * 60)

for w, (p, log) in enumerate(procs):
    log.close()
    print(f"worker {w} exit code {p.returncode}")
assert all(p.returncode == 0 for p, _ in procs), "A worker failed: read /kaggle/working/worker*.log"

[worker 0]
   Worker 0/2: recipe lr1e5_ep2, jobs [(10000, 44), (20000, 44), (40000, 42), (40000, 44)]. Validation 1000, test 600. Device cuda, mixed precision True.
     zero-shot validation accuracy 0.9300, macro-F1 0.9298
   scale_n10000_seed44: 10000 training reviews, seed 44
[worker 1]
   Worker 1/2: recipe lr1e5_ep2, jobs [(10000, 42), (10000, 43), (20000, 42), (20000, 43), (40000, 43)]. Validation 1000, test 600. Device cuda, mixed precision True.
     zero-shot validation accuracy 0.9300, macro-F1 0.9298
   scale_n10000_seed42: 10000 training reviews, seed 42
------------------------------------------------------------
[worker 0]
     zero-shot validation accuracy 0.9300, macro-F1 0.9298
   scale_n10000_seed44: 10000 training reviews, seed 44
     epoch 1/2: train loss 0.2127, validation accuracy 0.9350, macro-F1 0.9350
[worker 1]
     zero-shot validation accuracy 0.9300, macro-F1 0.9298
   scale_n10000_seed42: 10000 training reviews, seed 42
     epoch 1/2: train loss 0.2158, 

## 7. Select, re-score and log to MLflow

Picks the best run on **validation only**, keeps it as `models/nlptown_finetuned_v2`, scores the test set once more through the saved model (that is the model the API would serve), prints the tables, and replays every run into MLflow. The per-seed test scores are shown for the spread, but nothing is chosen with them.

Run this cell once. If you run it a second time, delete `/kaggle/working/scaling_mlflow.db` first or you will get duplicate runs.

In [9]:
TRACKING_URI = "sqlite:////kaggle/working/scaling_mlflow.db"
!python -m src.finetune_scaling summarize --tracking-uri $TRACKING_URI

Loading weights: 100%|██████████████████████| 201/201 [00:00<00:00, 2680.36it/s]

All runs
                run  train_size  seed  best_epoch  val_accuracy  val_macro_f1  test_accuracy  test_macro_f1  train_minutes
scale_n10000_seed42       10000    42           1        0.9440        0.9440         0.9433         0.9433         4.7000
scale_n10000_seed43       10000    43           1        0.9360        0.9360         0.9383         0.9383         4.7000
scale_n10000_seed44       10000    44           1        0.9350        0.9350         0.9283         0.9283         4.6000
scale_n20000_seed42       20000    42           2        0.9420        0.9420         0.9483         0.9483         9.4000
scale_n20000_seed43       20000    43           2        0.9460        0.9460         0.9450         0.9450         9.4000
scale_n20000_seed44       20000    44           2        0.9470        0.9470         0.9433         0.9433         9.2000
scale_n40000_seed42       40000    42           

## 8. Look at the logs (from MLflow)

In [10]:
import mlflow
import pandas as pd

mlflow.set_tracking_uri(TRACKING_URI)
runs = mlflow.search_runs(experiment_names=["reviewpulse-bert"])
wanted = ("tags.mlflow.runName", "params.train_size", "params.seed", "metrics.best_val", "metrics.test_", "metrics.best_epoch", "metrics.train_seconds")
columns = [c for c in runs.columns if c.startswith(wanted)]
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
runs[columns].sort_values("tags.mlflow.runName").round(4)

,metrics.test_accuracy,metrics.test_macro_f1,metrics.best_val_macro_f1,metrics.best_val_accuracy,metrics.best_epoch,metrics.train_seconds,params.train_size,params.seeds_per_size,params.seed,tags.mlflow.runName
9,0.9433,0.9433,0.944,0.944,1.0,279.5182,10000,None,42,scale_n10000_seed42
8,0.9383,0.9383,0.936,0.936,1.0,283.9972,10000,None,43,scale_n10000_seed43
7,0.9283,0.9283,0.935,0.935,1.0,277.7236,10000,None,44,scale_n10000_seed44
6,0.9483,0.9483,0.942,0.942,2.0,564.4112,20000,None,42,scale_n20000_seed42
5,0.9450,0.9450,0.946,0.946,2.0,563.6629,20000,None,43,scale_n20000_seed43
4,0.9433,0.9433,0.947,0.947,2.0,550.9657,20000,None,44,scale_n20000_seed44
3,0.9417,0.9417,0.951,0.951,2.0,1095.1375,40000,None,42,scale_n40000_seed42
2,0.9517,0.9517,0.952,0.952,2.0,1119.7952,40000,None,43,scale_n40000_seed43
1,0.9383,0.9383,0.949,0.949,2.0,1095.1373,40000,None,44,scale_n40000_seed44
0,0.9517,0.9517,NaN,NaN,NaN,NaN,40000,"[3, 3, 3]",43,scaling_selected_scale_n40000_seed43


## 9. Package the results

Creates, in `/kaggle/working`:
- `scaling_mlflow.db` (already there): every run with per-epoch curves
- `nlptown_finetuned_v2.zip`: the selected model (about 650 MB)
- `bert_finetuned_v2_test.csv` and `bert_finetuned_v2_test_metrics.json`: its test predictions and metrics
- `scaling_results.zip`: per-run test predictions and probabilities, `runs_table.csv`, `by_size.csv`, `report.json`
- `worker0.log`, `worker1.log`

Download them from the **Output** tab on the right (use **Save Version** to keep them after the session ends).

In [11]:
import shutil

shutil.make_archive("/kaggle/working/nlptown_finetuned_v2", "zip", "models", "nlptown_finetuned_v2")
shutil.make_archive("/kaggle/working/scaling_results", "zip", "data/predictions", "scaling")
for name in ["bert_finetuned_v2_test.csv", "bert_finetuned_v2_test_metrics.json"]:
    shutil.copy(f"data/predictions/{name}", "/kaggle/working/")
# the model folders are big and already zipped: remove the copies so Save Version stays small
shutil.rmtree("models/scaling_runs", ignore_errors=True)
!ls -lh /kaggle/working/*.zip /kaggle/working/*.db /kaggle/working/*.csv /kaggle/working/*.json /kaggle/working/*.log

-rw-r--r-- 1 root root  27K Nov 10 13:28 /kaggle/working/bert_finetuned_v2_test.csv
-rw-r--r-- 1 root root  304 Nov 10 13:28 /kaggle/working/bert_finetuned_v2_test_metrics.json
-rw-r--r-- 1 root root 594M Nov 10 13:28 /kaggle/working/nlptown_finetuned_v2.zip
-rw-r--r-- 1 root root 1.3M Nov 10 13:27 /kaggle/working/scaling_mlflow.db
-rw-r--r-- 1 root root 365K Nov 10 13:28 /kaggle/working/scaling_results.zip
-rw-r--r-- 1 root root 3.2K Nov 10 13:26 /kaggle/working/worker0.log
-rw-r--r-- 1 root root 3.9K Nov 10 13:23 /kaggle/working/worker1.log


## After the run
1. Download the files above.
2. On your laptop: unzip the model into `models/` (you get `models/nlptown_finetuned_v2/`), put the two `bert_finetuned_v2_test*` files in `data/predictions/`, and unzip `scaling_results.zip` into `data/predictions/`.
3. View the curves: `mlflow ui --backend-store-uri sqlite:///scaling_mlflow.db`
4. Send back the summary printed by cell 7 and the results table, so the slides and the paired tests against the earlier models can be updated.